In [ ]:
import cv2
import numpy as np
from pathlib import Path

img_ori_path=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-1.jpg")
img_a_path=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-2.jpg")
img_b_path=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-3.jpg")
save_path1=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-4.jpg")
save_path2=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-5.jpg")
save_path3=Path(r"C:\Users\zaoan\Desktop\CVexp\image\exp3-6.jpg")

img_ori=cv2.imread(str(img_ori_path)) # 原图
img_a=cv2.imread(str(img_a_path)) # 拼接的图片1
img_b=cv2.imread(str(img_b_path)) # 拼接的图片2

# 转为灰度图
gray_a=cv2.cvtColor(img_a,cv2.COLOR_BGR2GRAY)
gray_b=cv2.cvtColor(img_b,cv2.COLOR_BGR2GRAY)

# 初始化SIFT并提取特征
sift=cv2.SIFT_create()
kp_a,des_a=sift.detectAndCompute(gray_a,None) # a图片的特征点与128维向量描述
kp_b,des_b=sift.detectAndCompute(gray_b,None)
print("a图片的特征点与128维向量描述：", kp_a,des_a)
print("b图片的特征点与128维向量描述：", kp_b,des_b)

# 使用FLANN快速近似最近邻搜索库匹配器进行特征匹配 ,在大量向量里快速找“最相似向量”的工具。
FLANN_INDEX_KDTREE = 1 # 让 FLANN 用 KDTree 这种索引结构来加速搜索
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5) # 建索引时，用 KDTree建 5 棵树。
search_params = dict(checks=50) # 检查次数越多,匹配越准确但速度越慢。搜索最近邻时，检查 50 个候选节点。
flann = cv2. FlannBasedMatcher(index_params, search_params) # 创建一个 FLANN 匹配器对象。
matches = flann.knnMatch(des_a,des_b, k=2) # k=2表示每个特征点返回2个最佳匹配  
print("matches:",matches)

# 应用Lowe's比率测试筛选优质匹配点
good_matches = [] # m = 最近的那个匹配。n = 第二近的匹配。m.distance = 最近距离（越小越像）。n.distance = 第二近距离
for m, n in matches:
    if m.distance<0.7*n.distance: # 比率阈值通常取0.7-0.8.最近距离小于第二近距离的 0.7 倍，才保留。阈值越小，筛得越狠，留下的匹配越少但越可靠；
        good_matches.append(m)
print("good matches:",good_matches)

# 绘制匹配的SIFT关键点，NOT_DRAW_SINGLE_POINTS：只画有匹配的点，不画孤立点，图更干净。线乱飞、交叉严重 → 误匹配多，拼接可能失败。线大致平行、走向一致 → 匹配质量好
matched_keypoints_img = cv2.drawMatches(img_a, kp_a, img_b, kp_b, good_matches,None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

# 提取匹配点的坐标
src_pts = np.float32([kp_b[m.trainIdx].pt for m in good_matches]).reshape(-1,1,2) # 图像b的关键点
dst_pts = np.float32([kp_a[m.queryIdx].pt for m in good_matches]).reshape(-1,1,2) # 图像a的关键点

# 使用RANSAC算法估计单应矩阵(透视变换矩阵)。随机抽 4 对点算一个 H，看有多少点符合；重复很多次，选符合点最多的那个 H
H, _= cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0) # 5.0：重投影误差阈值，单位像素。变换后误差小于 5 像素算“符合”（inlier），否则算 outlier

# 获取输入图像尺寸
h_a, w_a = img_a. shape[:2]
h_b, w_b = img_b. shape[:2]

# 计算图像b变换后的四个角点坐标
pts = np.float32([[0, 0], [0, h_b], [w_b, h_b], [w_b, 0]]).reshape(-1, 1, 2)
dst_corners = cv2.perspectiveTransform(pts, H)

# 确定拼接后图像的最终尺寸(包含所有像素)
all_corners = np.concatenate([dst_corners, np.float32([[0,0], [w_a,0], [w_a,h_a],
[0,h_a]]).reshape(-1,1,2)], axis=0)
[x_min, y_min] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[x_max, y_max] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

# 创建平移矩阵,确保所有像素都在可见区域内
translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float32)

# 对图像b进行透视变换和平移
stitched_img = cv2.warpPerspective(img_b,translation_matrix @ H,(x_max-x_min, y_max-y_min))

# 将图像a复制到拼接结果的对应位置
stitched_img[-y_min : h_a - y_min, -x_min : w_a - x_min] = img_a

# 显示匹配关键点和拼接结果
cv2.imshow('Matched SIFT Keypoints', matched_keypoints_img)
cv2.imshow('Stitched Result', stitched_img)
cv2.waitKey(0) #等待用户按键(按任意键关闭窗口)
cv2.destroyAllWindows() #关闭所有显示窗口

# 保存拼接结果
cv2.imwrite(str(save_path1), stitched_img)
print("拼接完成,结果已保存为 exp3-4.jpg")
cv2.imwrite(str(save_path2), matched_keypoints_img)
print("拼接完成,匹配关键点结果已保存为 exp3-5.jpg")


# 新增：输出 原图 vs 拼接结果 的对比图（含属性信息）
def print_image_info(tag, img, file_path=None):
    """在控制台打印图像属性"""
    print(f"\n【{tag}】")
    print(f"  尺寸(高×宽) : {img.shape[0]} × {img.shape[1]}")
    print(f"  shape       : {img.shape}")
    print(f"  通道数      : {img.shape[2] if img.ndim == 3 else 1}")
    print(f"  dtype       : {img.dtype}")
    print(f"  像素总数    : {img.shape[0] * img.shape[1]:,}")
    print(f"  内存占用    : {img.nbytes / 1024:.2f} KB")
    if file_path is not None and Path(file_path).exists():
        size_kb = Path(file_path).stat().st_size / 1024
        print(f"  文件大小    : {size_kb:.2f} KB")
        print(f"  文件路径    : {file_path}")


print("\n" + "=" * 55)
print("          原图 与 拼接结果 属性对比")
print("=" * 55)
print_image_info("原图 img_ori", img_ori, img_ori_path)
print_image_info("拼接结果 stitched_img", stitched_img, save_path1)


def make_comparison(images, labels, target_h=500, gap=25,
                    header_h=75, footer_h=50):
    """
    把多张图缩放到相同高度后水平拼接：
      - 每张图上方写：名称 + 原始尺寸
      - 底部写一行汇总属性
    """
    # 1) 统一高度，保持宽高比
    scaled = []
    for img, label in zip(images, labels):
        h, w = img.shape[:2]
        new_w = max(1, int(w * target_h / h))
        im = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_AREA)
        scaled.append((im, label, (w, h)))

    # 2) 创建白色画布
    total_w = sum(im.shape[1] for im, _, _ in scaled) + gap * (len(scaled) - 1)
    canvas = np.ones((header_h + target_h + footer_h, total_w, 3),
                     dtype=np.uint8) * 255

    # 3) 依次贴图 + 写标签
    x = 0
    for im, label, (orig_w, orig_h) in scaled:
        h, w = im.shape[:2]
        canvas[header_h:header_h + h, x:x + w] = im

        # 画个浅灰边框，方便区分两张图
        cv2.rectangle(canvas, (x, header_h),
                      (x + w - 1, header_h + h - 1),
                      (200, 200, 200), 1)

        # 图像名称
        cv2.putText(canvas, label, (x + 8, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85,
                    (0, 0, 0), 2, cv2.LINE_AA)

        # 原始尺寸（注意是原图尺寸，不是缩放后尺寸）
        cv2.putText(canvas, f"orig {orig_w}x{orig_h}", (x + 8, 62),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                    (0, 0, 180), 1, cv2.LINE_AA)

        x += w + gap

    # 4) 底部汇总一行属性
    info = (f"Original: {img_ori.shape[1]}x{img_ori.shape[0]}  |  "
            f"Stitched: {stitched_img.shape[1]}x{stitched_img.shape[0]}  |  "
            f"Channels: {stitched_img.shape[2] if stitched_img.ndim == 3 else 1}")
    cv2.putText(canvas, info, (10, canvas.shape[0] - 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                (0, 100, 0), 2, cv2.LINE_AA)

    return canvas


# 生成对比图
comparison_img = make_comparison(
    [img_ori, stitched_img],
    ["Original", "Stitched Result"],
    target_h=500
)

cv2.imshow('Original vs Stitched', comparison_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# 保存对比图
cv2.imwrite(str(save_path3), comparison_img)
print(f"\n对比图已保存为: {save_path3}")

a图片的特征点与128维向量描述： (< cv2.KeyPoint 0000022B7B15ECD0>, < cv2.KeyPoint 0000022B5B6E91A0>, < cv2.KeyPoint 0000022B001A6100>, < cv2.KeyPoint 0000022B001A4810>, < cv2.KeyPoint 0000022B00046BE0>, < cv2.KeyPoint 0000022B00045470>, < cv2.KeyPoint 0000022B00044030>, < cv2.KeyPoint 0000022B00044060>, < cv2.KeyPoint 0000022B00044090>, < cv2.KeyPoint 0000022B000440C0>, < cv2.KeyPoint 0000022B000440F0>, < cv2.KeyPoint 0000022B00044120>, < cv2.KeyPoint 0000022B00044150>, < cv2.KeyPoint 0000022B00044180>, < cv2.KeyPoint 0000022B000441B0>, < cv2.KeyPoint 0000022B000441E0>, < cv2.KeyPoint 0000022B00044210>, < cv2.KeyPoint 0000022B00044240>, < cv2.KeyPoint 0000022B00044270>, < cv2.KeyPoint 0000022B000442A0>, < cv2.KeyPoint 0000022B000442D0>, < cv2.KeyPoint 0000022B00044300>, < cv2.KeyPoint 0000022B00044330>, < cv2.KeyPoint 0000022B00044360>, < cv2.KeyPoint 0000022B000443C0>, < cv2.KeyPoint 0000022B000443F0>, < cv2.KeyPoint 0000022B000444B0>, < cv2.KeyPoint 0000022B000444E0>, < cv2.KeyPoint 0000022B000445